<a href="https://colab.research.google.com/github/Xyrfo/Pembelajaran-Mesin/blob/main/JS03/TugasLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv("wbc.csv")

df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")

y = df["diagnosis"]
X = df.drop(columns=["diagnosis"])

print(X.shape)
X.head()

(569, 30)


,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [ ]:
le = LabelEncoder()
y = le.fit_transform(y)
print(le.classes_)

['B' 'M']


In [ ]:
num_cols = X.columns.tolist()
scaler = StandardScaler()

In [ ]:
selector_filter = SelectKBest(score_func=f_classif, k=10)

In [ ]:
pipe_filter = Pipeline([
    ("scaler", scaler),
    ("sel", selector_filter),
    ("clf", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

pipe_filter.fit(X_train, y_train)
pred = pipe_filter.predict(X_test)

print("=== Filter (ANOVA) + LR ===")
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

=== Filter (ANOVA) + LR ===
Accuracy: 0.956140350877193
              precision    recall  f1-score   support

           0       0.95      0.99      0.97        72
           1       0.97      0.90      0.94        42

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114



In [ ]:
results = []

for k in range(2, X.shape[1] + 1):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("sel", SelectKBest(score_func=f_classif, k=k)),
        ("clf", LogisticRegression(max_iter=1000))
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, pred)
    results.append((k, acc))

results_df = pd.DataFrame(results, columns=["k", "accuracy"]).sort_values("accuracy", ascending=False)
print(results_df.head(10))

best_k = int(results_df.iloc[0]["k"])
print("\nJumlah fitur terbaik (k):", best_k)

     k  accuracy
12  14  0.982456
17  19  0.982456
18  20  0.982456
16  18  0.982456
10  12  0.973684
20  22  0.973684
19  21  0.973684
14  16  0.973684
15  17  0.973684
13  15  0.973684

Jumlah fitur terbaik (k): 14


In [ ]:
pipe_best = Pipeline([
    ("scaler", StandardScaler()),
    ("sel", SelectKBest(score_func=f_classif, k=best_k)),
    ("clf", LogisticRegression(max_iter=1000))
])
pipe_best.fit(X_train, y_train)

sel = pipe_best.named_steps["sel"]
mask = sel.get_support()
selected_names = X.columns[mask]
selected_scores = sel.scores_[mask]

top = sorted(zip(selected_names, selected_scores), key=lambda t: t[1], reverse=True)
print("Fitur terpilih (urut skor ANOVA tertinggi):")
for name, score in top:
    print(f"- {name}: {score:.2f}")

Fitur terpilih (urut skor ANOVA tertinggi):
- concave points_worst: 733.72
- perimeter_worst: 717.25
- radius_worst: 692.86
- concave points_mean: 684.53
- perimeter_mean: 548.41
- area_worst: 522.19
- radius_mean: 511.27
- area_mean: 444.86
- concavity_mean: 397.59
- concavity_worst: 319.51
- compactness_mean: 263.56
- compactness_worst: 238.20
- radius_se: 205.43
- perimeter_se: 193.17
